# PREMIER — Scraper pubs Instagram
Trouve les artistes qui **payent pour promouvoir leur musique** sur Instagram.

**Résultat :** liste de `@usernames` uniquement.

---
### Comment obtenir ton token Meta :
1. Va sur [developers.facebook.com/tools/explorer](https://developers.facebook.com/tools/explorer)
2. Connecte-toi avec Facebook
3. Clique **Generate Access Token**
4. Copie le token ci-dessous


In [ ]:
# ✏️ COLLE TON TOKEN ICI
TOKEN = "EAA..."

# ✏️ CHOISIS TES PAYS (codes ISO)
PAYS = ["FR", "BE", "CH", "CI", "SN", "CM", "MA"]

# ✏️ MOTS-CLÉS À RECHERCHER (tu peux en ajouter)
MOTS_CLES = ["single", "clip", "spotify", "disponible", "out now", "album", "ep"]

In [ ]:
import requests, json, re, time

API = "https://graph.facebook.com/v19.0/ads_archive"
usernames = set()

def extraire_username(ad):
    snap = ad.get("ad_snapshot_url", "")
    m = re.search(r"ig_username=([^&]+)", snap)
    if m:
        return "@" + m.group(1)
    nom = ad.get("page_name", "")
    if nom:
        slug = re.sub(r"[^a-zA-Z0-9._]", "", nom.lower())
        if slug:
            return "@" + slug
    return None

for mot in MOTS_CLES:
    print(f"🔍 Recherche : '{mot}'...")
    params = {
        "access_token": TOKEN,
        "ad_type": "ALL",
        "ad_active_status": "ACTIVE",
        "search_terms": mot,
        "ad_reached_countries": json.dumps(PAYS),
        "fields": "page_name,publisher_platforms,ad_snapshot_url",
        "limit": 100,
    }
    try:
        r = requests.get(API, params=params, timeout=15)
        data = r.json()
        if "error" in data:
            print(f"  ❌ Erreur API : {data['error']['message']}")
            break
        ads = data.get("data", [])
        trouves = 0
        for ad in ads:
            if "instagram" not in ad.get("publisher_platforms", []):
                continue
            u = extraire_username(ad)
            if u:
                usernames.add(u)
                trouves += 1
        print(f"  ✅ {trouves} compte(s) trouvé(s)")
    except Exception as e:
        print(f"  ❌ {e}")
    time.sleep(1)

print(f"\n{'='*40}")
print(f"TOTAL : {len(usernames)} comptes uniques")
print('='*40)

In [ ]:
# Affichage de la liste finale
liste = sorted(usernames)
for u in liste:
    print(u)

In [ ]:
# Export CSV (téléchargeable depuis Colab)
import csv
from google.colab import files

with open("PREMIER_usernames.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["username"])
    for u in liste:
        w.writerow([u])

files.download("PREMIER_usernames.csv")
print("✅ Fichier téléchargé !")